In [3]:
import os
import pickle
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# ================= 配置区 =================
# 支持传入多个归档目录进行跨版本合并
ROOT_DIRS = ["../result/archive-100", "../result/archive-101"] 
OUTPUT_FILE_LOC = "./result_101/experiment_results_PowerBI.csv"

# 定义需要保留的严格 Budget 目标值 (6, 7, 8~15)
TARGET_BUDGETS = {6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0}
# ==========================================

def process_experiment_data(root_paths):
    data_records = []
    pckl_files = []
    
    # 聚拢多个源目录的文件
    for r in root_paths:
        path_obj = Path(r)
        if path_obj.exists():
            pckl_files.extend(list(path_obj.rglob("*.pckl")))
        else:
            print(f"⚠️ 警告: 找不到目录 {r}")
            
    print(f"🔍 总共寻找到 {len(pckl_files)} 个 pickle 文件，开始智能合并解析...")
    
    for pckl_file in tqdm(pckl_files, desc="Parsing Files"):
        try:
            # 1. 从目录结构解析环境参数
            seed_str = pckl_file.parent.name
            num_str = pckl_file.parent.parent.name
            task_name = pckl_file.parent.parent.parent.name
            
            # --- 过滤：无视 caltech n=767 ---
            if task_name == 'caltech' and num_str == '767':
                continue
            
            # 识别当前文件属于哪个 archive (用于判定新旧版本)
            in_archive_100 = 'archive-100' in pckl_file.parts
            in_archive_101 = 'archive-101' in pckl_file.parts
            
            # 2. 从文件名解析策略参数
            filename = pckl_file.stem
            parts = filename.split('-')
            
            if not parts:
                continue
                
            alg = parts[0]
            
            # 建立基础字典
            record = {
                'Task': task_name,
                'Ground_Size': int(num_str),
                'Seed': int(seed_str),
                'Algorithm': None,
                'Strategy': None,
                'UB': None,
                'D': None,
                'Budget': None,
                'Alpha': None,
                'Model': None,
            }

            # 3. 跨归档的分流解析逻辑
            if in_archive_100:
                # 剔除 100 目录下的旧版/错误版 EfficientBFS
                if alg == 'EfficientBFS':
                    continue
                    
                # 仅保留基准算法 BFSTC 和 Efficient
                if alg in ['Efficient', 'BFSTC'] and len(parts) >= 6:
                    budget = float(parts[3])
                    # 容差判定，过滤不需要的 budget
                    if not any(abs(budget - t) < 1e-4 for t in TARGET_BUDGETS):
                        continue
                        
                    record['Algorithm'] = alg
                    record['UB'] = parts[1]
                    record['D'] = parts[2]
                    record['Budget'] = budget
                    record['Alpha'] = float(parts[4])
                    record['Model'] = "-".join(parts[5:])
                else:
                    continue
                    
            elif in_archive_101:
                # 原有的逻辑：抓取新版修复后的 Standard (Traditional)
                if alg == 'Exp' and parts[1] == 'Standard' and len(parts) >= 5:
                    budget = float(parts[3])
                    # 容差判定
                    if not any(abs(budget - t) < 1e-4 for t in TARGET_BUDGETS):
                        continue
                        
                    # 将其伪装/对齐回原版命名系统
                    record['Algorithm'] = 'EfficientBFS'
                    record['Strategy'] = 'Standard'  
                    record['UB'] = parts[2]
                    record['Budget'] = budget
                    record['Alpha'] = float(parts[4])
                
                # 新增逻辑：处理复制进来的 EfficientBFS-traditional 数据，并归类为 simple 算法
                elif alg == 'EfficientBFS' and parts[1] == 'traditional' and len(parts) >= 6:
                    budget = float(parts[4])
                    # 容差判定
                    if not any(abs(budget - t) < 1e-4 for t in TARGET_BUDGETS):
                        continue
                    
                    # 划分为名为 simple 的新算法
                    record['Algorithm'] = 'simple'
                    record['Strategy'] = parts[1]
                    record['UB'] = parts[2]
                    record['D'] = parts[3]
                    record['Budget'] = budget
                    record['Alpha'] = float(parts[5])
                else:
                    continue
            else:
                continue

            # 4. 读取 Pickle 内容
            with open(pckl_file, 'rb') as f:
                res = pickle.load(f)
                
            # 5. 处理 Time_s 和 TLE
            raw_time = res.get('time', None)
            raw_tle = res.get('TLE', False)
            
            if raw_time is not None and raw_time >= 5000:
                raw_time = 5000
                raw_tle = True
                
            # 6. 组装核心指标
            record.update({
                'Objective_f(S)': res.get('f(S)', None),
                'Cost_c(S)': res.get('c(S)', None),
                'Time_s': raw_time,
                'Node_Count': res.get('node_count', None),
                'Open_List_Count': res.get('open_list_count', None),
                'TLE': raw_tle,
                'Solution_Set_Size': len(res.get('S', [])) if 'S' in res else 0 
            })
            
            data_records.append(record)
            
        except Exception as e:
            print(f"❌ 解析出错 {pckl_file.name}: {e}")

    # 7. 转换为 DataFrame 并导出
    df = pd.DataFrame(data_records)
    
    if not df.empty:
        # 按照任务、算法、预算、种子进行多级排序，确保数据规整
        df.sort_values(by=['Task', 'Algorithm', 'Budget', 'Seed'], inplace=True)
        
        # 自动创建输出目录
        Path(OUTPUT_FILE_LOC).parent.mkdir(parents=True, exist_ok=True)
        
        df.to_csv(OUTPUT_FILE_LOC, index=False, encoding='utf-8-sig')
        print(f"\n✅ 数据处理完毕！共提取 {len(df)} 条有效记录。")
        print(f"💾 已保存至: {OUTPUT_FILE_LOC}")
    else:
        print("\n⚠️ 未提取到任何有效数据，请检查 ROOT_DIRS 路径和解析规则。")
        
    return df

# 执行提取
df_results = process_experiment_data(ROOT_DIRS)

# 预览前 5 行数据
if not df_results.empty:
    print(df_results.head())

🔍 总共寻找到 916 个 pickle 文件，开始智能合并解析...


Parsing Files: 100%|██████████| 916/916 [00:00<00:00, 9412.06it/s]


✅ 数据处理完毕！共提取 209 条有效记录。
💾 已保存至: ./result_101/experiment_results_PowerBI.csv
     Task  Ground_Size  Seed Algorithm Strategy   UB  D  Budget  Alpha  \
6   adult          111     0     BFSTC     None  ub0  d     6.0   0.95   
10  adult          111     0     BFSTC     None  ub2  d     6.0   0.95   
7   adult          111     0     BFSTC     None  ub0  d     7.0   0.95   
11  adult          111     0     BFSTC     None  ub2  d     7.0   0.95   
8   adult          111     0     BFSTC     None  ub0  d     8.0   0.95   

                          Model  Objective_f(S)  Cost_c(S)       Time_s  \
6   AdultIncomeFeatureSelection        7.206555   5.836918  5000.000000   
10  AdultIncomeFeatureSelection        7.206555   5.836918    10.346726   
7   AdultIncomeFeatureSelection        7.206555   6.999571  5000.000000   
11  AdultIncomeFeatureSelection        7.206555   6.999571    10.518110   
8   AdultIncomeFeatureSelection        8.120308   7.606156  5000.000000   

    Node_Count  Open_List_C